# Problema 21: Combinacion Modal CQC vs SRSS

## Relacion 3 - Analisis Sismico Espectral

**Objetivo:** Comparar los metodos de combinacion modal SRSS (Square Root of Sum of Squares) y CQC (Complete Quadratic Combination) para el analisis sismico de estructuras.

### Enunciado

Se tiene un edificio de 3 plantas uniforme con:
- Masa por planta: $m = 1000$ kg
- Rigidez por entreplanta: $k = 200$ kN/m
- Amortiguamiento critico: $\zeta = 5\%$

Se dispone de los siguientes valores del espectro de aceleraciones:
- $S_a(T_1) = 2.5$ m/s$^2$
- $S_a(T_2) = 3.0$ m/s$^2$
- $S_a(T_3) = 2.8$ m/s$^2$

Se pide calcular los desplazamientos maximos mediante SRSS y CQC.

## Datos del Problema

| Parametro | Valor | Unidad |
|-----------|-------|--------|
| $n$ | 3 | plantas |
| $m$ | 1000 | kg |
| $k$ | 200 | kN/m |
| $\zeta$ | 5 | % |
| $S_a(T_1)$ | 2.5 | m/s$^2$ |
| $S_a(T_2)$ | 3.0 | m/s$^2$ |
| $S_a(T_3)$ | 2.8 | m/s$^2$ |

In [ ]:
# Importacion de librerias
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh

# Configuracion de graficos
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11

# Datos del problema
n = 3            # Numero de plantas
m = 1000.0       # kg por planta
k = 200e3        # N/m por entreplanta
zeta = 0.05      # 5% amortiguamiento

# Valores espectrales dados
Sa = np.array([2.5, 3.0, 2.8])  # m/s^2

print("DATOS DEL PROBLEMA")
print("="*40)
print(f"Numero de plantas: n = {n}")
print(f"Masa por planta: m = {m:.0f} kg")
print(f"Rigidez por entreplanta: k = {k/1e3:.0f} kN/m")
print(f"Amortiguamiento: zeta = {zeta*100:.0f}%")
print(f"\nEspectro de aceleraciones:")
for i, sa in enumerate(Sa):
    print(f"  Sa(T_{i+1}) = {sa} m/s^2")

## 1. Construccion del Sistema y Analisis Modal

In [ ]:
def shear_building_uniforme(n, m, k):
    """Construye M y K para un shear building uniforme."""
    M = np.eye(n) * m
    K = np.zeros((n, n))
    for i in range(n):
        if i == 0:
            K[i, i] = 2 * k
        elif i == n-1:
            K[i, i] = k
        else:
            K[i, i] = 2 * k
        if i < n-1:
            K[i, i+1] = -k
            K[i+1, i] = -k
    return M, K

# Construir sistema
M, K = shear_building_uniforme(n, m, k)

# Resolver problema de valores propios
eigenvalues, eigenvectors = eigh(K, M)
omega = np.sqrt(eigenvalues)
freq = omega / (2 * np.pi)
T = 1 / freq
phi = eigenvectors  # Modos normalizados masa-unidad

print("PROPIEDADES MODALES")
print("="*50)
print(f"{'Modo':<6} {'omega (rad/s)':<15} {'f (Hz)':<12} {'T (s)':<10}")
print("-"*50)
for j in range(n):
    print(f"{j+1:<6} {omega[j]:<15.2f} {freq[j]:<12.2f} {T[j]:<10.3f}")

# Ratios de frecuencia
print("\nRatios de frecuencia:")
for i in range(n-1):
    print(f"  omega_{i+2}/omega_{i+1} = {omega[i+1]/omega[i]:.2f}")

## 2. Factores de Participacion Modal

Para excitacion en la base con vector de influencia $\mathbf{r} = [1, 1, 1]^T$:

$$\Gamma_n = \frac{\boldsymbol{\phi}_n^T \mathbf{M} \mathbf{r}}{\boldsymbol{\phi}_n^T \mathbf{M} \boldsymbol{\phi}_n}$$

In [ ]:
# Vector de influencia
r = np.ones(n)

# Calcular factores de participacion
Gamma = np.zeros(n)
for j in range(n):
    L_n = phi[:, j] @ M @ r
    m_n = phi[:, j] @ M @ phi[:, j]  # = 1 para modos normalizados
    Gamma[j] = L_n / m_n

print("FACTORES DE PARTICIPACION MODAL")
print("="*40)
for j in range(n):
    print(f"Gamma_{j+1} = {Gamma[j]:.4f}")

## 3. Contribuciones Modales Maximas

El desplazamiento espectral se obtiene de:
$$S_d = \frac{S_a}{\omega^2}$$

La contribucion modal maxima para cada DOF es:
$$x_{j,n}^{max} = \Gamma_n \cdot \phi_{j,n} \cdot S_d(T_n)$$

In [ ]:
# Desplazamientos espectrales
Sd = Sa / omega**2

print("DESPLAZAMIENTOS ESPECTRALES")
print("="*40)
for j in range(n):
    print(f"Sd(T_{j+1}) = {Sd[j]*1000:.3f} mm")

# Contribuciones modales para cada DOF
x_modal = np.zeros((n, n))  # x_modal[dof, modo]
for modo in range(n):
    x_modal[:, modo] = Gamma[modo] * phi[:, modo] * Sd[modo]

print("\nCONTRIBUCIONES MODALES MAXIMAS")
print("="*50)
print(f"{'DOF':<8}", end="")
for j in range(n):
    print(f"{'Modo '+str(j+1):<14}", end="")
print()
print("-"*50)
for i in range(n):
    print(f"P{i+1:<7}", end="")
    for j in range(n):
        print(f"{x_modal[i,j]*1000:<14.3f}", end="")
    print(" mm")

## 4. Metodo SRSS (Square Root of Sum of Squares)

La combinacion SRSS asume que los maximos modales son estadisticamente independientes:

$$x_j^{max} = \sqrt{\sum_{n=1}^{N} (x_{j,n}^{max})^2}$$

Este metodo es adecuado cuando las frecuencias naturales estan bien separadas ($\omega_i/\omega_j > 1.5$).

In [ ]:
# Combinacion SRSS
x_srss = np.sqrt(np.sum(x_modal**2, axis=1))

print("COMBINACION SRSS")
print("="*40)
print("x_SRSS = sqrt(Sum x_r^2)")
print()
for i in range(n):
    terms = " + ".join([f"({x_modal[i,j]*1000:.3f})^2" for j in range(n)])
    print(f"x_{i+1}_SRSS = sqrt({terms})")
    print(f"         = {x_srss[i]*1000:.3f} mm")
    print()

## 5. Coeficientes de Correlacion CQC

El metodo CQC considera la correlacion entre modos mediante el coeficiente de Der Kiureghian:

$$\rho_{ij} = \frac{8\sqrt{\zeta_i \zeta_j}(\zeta_i + \beta_{ij}\zeta_j)\beta_{ij}^{3/2}}{(1-\beta_{ij}^2)^2 + 4\zeta_i\zeta_j\beta_{ij}(1+\beta_{ij}^2) + 4(\zeta_i^2 + \zeta_j^2)\beta_{ij}^2}$$

Donde $\beta_{ij} = \omega_i/\omega_j$ (ratio de frecuencias).

In [ ]:
def coef_cqc(omega_i, omega_j, zeta_i, zeta_j):
    """
    Coeficiente de correlacion CQC (Der Kiureghian).
    """
    beta = omega_i / omega_j
    if beta > 1:
        beta = 1 / beta
    
    num = 8 * np.sqrt(zeta_i * zeta_j) * (zeta_i + beta * zeta_j) * beta**(3/2)
    den = ((1 - beta**2)**2 + 
           4 * zeta_i * zeta_j * beta * (1 + beta**2) + 
           4 * (zeta_i**2 + zeta_j**2) * beta**2)
    
    return num / den

# Calcular matriz de correlacion
rho = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        if i == j:
            rho[i, j] = 1.0
        else:
            rho[i, j] = coef_cqc(omega[i], omega[j], zeta, zeta)

print("MATRIZ DE CORRELACION CQC")
print("="*50)
print(f"{'':>10}", end="")
for j in range(n):
    print(f"{'Modo '+str(j+1):>12}", end="")
print()
print("-"*50)
for i in range(n):
    print(f"Modo {i+1}:", end="")
    for j in range(n):
        print(f"{rho[i,j]:>12.4f}", end="")
    print()

print("\nNOTA: Coeficientes fuera de la diagonal pequenos indican")
print("      que los modos estan bien separados en frecuencia.")

## 6. Metodo CQC (Complete Quadratic Combination)

La combinacion CQC incluye la correlacion entre modos:

$$x_j^{max} = \sqrt{\sum_{i=1}^{N}\sum_{k=1}^{N} \rho_{ik} \cdot x_{j,i}^{max} \cdot x_{j,k}^{max}}$$

Este metodo es mas preciso cuando las frecuencias estan proximas.

In [ ]:
# Combinacion CQC
x_cqc = np.zeros(n)
for dof in range(n):
    suma = 0
    for i in range(n):
        for j in range(n):
            suma += rho[i, j] * x_modal[dof, i] * x_modal[dof, j]
    x_cqc[dof] = np.sqrt(suma)

print("COMBINACION CQC")
print("="*40)
print("x_CQC = sqrt(Sum_i Sum_j rho_ij * x_i * x_j)")
print()
for i in range(n):
    print(f"x_{i+1}_CQC = {x_cqc[i]*1000:.3f} mm")

## 7. Comparacion de Resultados

In [ ]:
print("COMPARACION SRSS vs CQC")
print("="*55)
print(f"{'DOF':<8} {'SRSS (mm)':<14} {'CQC (mm)':<14} {'Diferencia (%)':<15}")
print("-"*55)
for i in range(n):
    dif = (x_cqc[i] - x_srss[i]) / x_srss[i] * 100
    print(f"P{i+1:<7} {x_srss[i]*1000:<14.3f} {x_cqc[i]*1000:<14.3f} {dif:<+15.2f}")

## 8. Visualizacion de Resultados

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Modos de vibracion
ax1 = axes[0, 0]
plantas = np.arange(n + 1)
colores = ['blue', 'red', 'green']
for j in range(n):
    modo_norm = phi[:, j] / np.max(np.abs(phi[:, j]))
    modo_plot = np.concatenate([[0], modo_norm])
    ax1.plot(modo_plot, plantas, '-o', color=colores[j], linewidth=2,
             markersize=10, label=f'Modo {j+1} (f={freq[j]:.2f} Hz)')

ax1.axvline(x=0, color='k', linestyle='--', linewidth=0.5)
ax1.set_xlabel('Desplazamiento normalizado')
ax1.set_ylabel('Planta')
ax1.set_yticks(plantas)
ax1.set_yticklabels(['Base', 'P1', 'P2', 'P3'])
ax1.legend(loc='upper left')
ax1.set_title('Modos de vibracion', fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-1.3, 1.3)

# Panel 2: Contribuciones modales por DOF
ax2 = axes[0, 1]
x_pos = np.arange(n)
width = 0.25
for j in range(n):
    ax2.bar(x_pos + j*width, np.abs(x_modal[:, j])*1000, width,
            label=f'Modo {j+1}', color=colores[j], alpha=0.8, edgecolor='black')

ax2.set_xlabel('Planta')
ax2.set_ylabel('Desplazamiento modal |x| (mm)')
ax2.set_xticks(x_pos + width)
ax2.set_xticklabels(['P1', 'P2', 'P3'])
ax2.legend()
ax2.set_title('Contribuciones modales', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Panel 3: Matriz de correlacion
ax3 = axes[1, 0]
im = ax3.imshow(rho, cmap='RdYlBu_r', vmin=0, vmax=1)
ax3.set_xticks(np.arange(n))
ax3.set_yticks(np.arange(n))
ax3.set_xticklabels([f'Modo {i+1}' for i in range(n)])
ax3.set_yticklabels([f'Modo {i+1}' for i in range(n)])
for i in range(n):
    for j in range(n):
        color = 'white' if rho[i,j] > 0.5 else 'black'
        ax3.text(j, i, f'{rho[i,j]:.3f}', ha='center', va='center', 
                 color=color, fontsize=11, fontweight='bold')
ax3.set_title('Matriz de correlacion CQC', fontweight='bold')
plt.colorbar(im, ax=ax3)

# Panel 4: Comparacion SRSS vs CQC
ax4 = axes[1, 1]
x_pos = np.arange(n)
width = 0.35
bars1 = ax4.bar(x_pos - width/2, x_srss*1000, width, label='SRSS', 
                color='steelblue', edgecolor='black')
bars2 = ax4.bar(x_pos + width/2, x_cqc*1000, width, label='CQC', 
                color='coral', edgecolor='black')

ax4.set_xlabel('Planta')
ax4.set_ylabel('Desplazamiento maximo (mm)')
ax4.set_xticks(x_pos)
ax4.set_xticklabels(['P1', 'P2', 'P3'])
ax4.legend()
ax4.set_title('Comparacion SRSS vs CQC', fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

# Etiquetas con diferencia
for i, (b1, b2) in enumerate(zip(bars1, bars2)):
    dif = (x_cqc[i] - x_srss[i]) / x_srss[i] * 100
    max_h = max(b1.get_height(), b2.get_height())
    ax4.text(i, max_h + 0.3, f'{dif:+.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## Interpretacion de Resultados

### Separacion de frecuencias
- Los ratios de frecuencia son $\omega_2/\omega_1 \approx 2.0$ y $\omega_3/\omega_2 \approx 1.3$
- Las frecuencias estan razonablemente separadas (ratio > 1.5 para los dos primeros modos)

### Coeficientes de correlacion
- Los coeficientes fuera de la diagonal son pequenos ($\rho_{12} \approx 0.01$, $\rho_{23} \approx 0.06$)
- Esto indica que los modos son practicamente independientes

### Comparacion SRSS vs CQC
- La diferencia entre SRSS y CQC es muy pequena (< 1%)
- Esto era esperado dado que las frecuencias estan bien separadas

### Recomendaciones de uso

**Usar CQC cuando:**
1. Frecuencias cercanas ($\omega_i/\omega_j < 1.5$)
2. Alto amortiguamiento (aumenta la correlacion)
3. Estructuras con modos de torsion acoplados
4. Edificios con simetria que produce frecuencias repetidas

**SRSS es suficiente cuando:**
1. Frecuencias bien separadas ($\omega_i/\omega_j > 1.5$)
2. Bajo amortiguamiento ($\zeta < 10\%$)
3. Estructuras regulares sin modos acoplados